# 1. Imports

In [3]:
import pandas as pd
import sqlite3

# 2. Laad de bronbestanden in

In [4]:
conn_crm = sqlite3.connect('CRM-data.sqlite')
conn_sales = sqlite3.connect('GO_SALES-data.sqlite')
conn_staff = sqlite3.connect('GO_STAFF-data.sqlite')

df_inventory = pd.read_csv('INVENTORY_LEVELS-data.csv', encoding='latin1')
df_forecast = pd.read_csv('PRODUCT_FORECAST-data.csv', encoding='latin1')
df_targets = pd.read_csv('SALES_TARGET-data.csv', encoding='latin1')

# 3. Transformeer de dimensies

In [5]:
query_product = """
    SELECT p.PRODUCT_NUMBER, p.PRODUCT_NAME, p.PRODUCT_TYPE_CODE, pt.PRODUCT_TYPE_EN,
           pt.PRODUCT_LINE_CODE, pl.PRODUCT_LINE_EN, p.PRODUCTION_COST, p.MARGIN
    FROM product p
    LEFT JOIN product_type pt ON p.PRODUCT_TYPE_CODE = pt.PRODUCT_TYPE_CODE
    LEFT JOIN product_line pl ON pt.PRODUCT_LINE_CODE = pl.PRODUCT_LINE_CODE
"""
dim_product = pd.read_sql_query(query_product, conn_sales)

query_customer = """
    SELECT rs.RETAILER_SITE_CODE, rs.RETAILER_CODE, rs.CITY, rs.REGION, rs.COUNTRY_CODE, c.COUNTRY
    FROM retailer_site rs
    LEFT JOIN country c ON rs.COUNTRY_CODE = c.COUNTRY_CODE
"""
dim_customer = pd.read_sql_query(query_customer, conn_sales)

query_staff = """
    SELECT s.SALES_STAFF_CODE, s.FIRST_NAME, s.LAST_NAME, s.POSITION_EN, sb.REGION AS BRANCH_REGION
    FROM sales_staff s
    LEFT JOIN sales_branch sb ON s.SALES_BRANCH_CODE = sb.SALES_BRANCH_CODE
"""
dim_staff = pd.read_sql_query(query_staff, conn_sales)

query_return_reason = "SELECT RETURN_REASON_CODE, RETURN_DESCRIPTION_EN FROM return_reason"
dim_return_reason = pd.read_sql_query(query_return_reason, conn_sales)

dim_date = pd.DataFrame({'DATE': pd.date_range(start='2020-01-01', end='2026-12-31')})
dim_date['DATE_ID'] = dim_date['DATE'].dt.strftime('%Y%m%d').astype(int)
dim_date['YEAR'] = dim_date['DATE'].dt.year
dim_date['MONTH'] = dim_date['DATE'].dt.month
dim_date['DAY'] = dim_date['DATE'].dt.day
dim_date['DATE_STR'] = dim_date['DATE'].dt.strftime('%Y-%m-%d')

# 4. Tranformeer de feiten

In [6]:
query_sales = """
    SELECT
        oh.ORDER_DATE,
        od.ORDER_DETAIL_CODE,
        oh.ORDER_NUMBER,
        od.PRODUCT_NUMBER,
        oh.RETAILER_SITE_CODE,
        oh.SALES_STAFF_CODE,
        od.QUANTITY,
        od.UNIT_COST,
        od.UNIT_SALE_PRICE,
        (od.QUANTITY * od.UNIT_SALE_PRICE) AS REVENUE,
        (od.QUANTITY * od.UNIT_COST) AS TOTAL_COST
    FROM order_details od
    JOIN order_header oh ON od.ORDER_NUMBER = oh.ORDER_NUMBER
"""
fact_sales = pd.read_sql_query(query_sales, conn_sales)
fact_sales['ORDER_DATE'] = pd.to_datetime(fact_sales['ORDER_DATE'])
fact_sales['DATE_ID'] = fact_sales['ORDER_DATE'].dt.strftime('%Y%m%d').astype(int)

query_returns = """
    SELECT
        ri.RETURN_CODE,
        ri.RETURN_DATE,
        ri.ORDER_DETAIL_CODE,
        ri.RETURN_REASON_CODE,
        ri.RETURN_QUANTITY,
        od.PRODUCT_NUMBER,
        oh.RETAILER_SITE_CODE
    FROM returned_item ri
    JOIN order_details od ON ri.ORDER_DETAIL_CODE = od.ORDER_DETAIL_CODE
    JOIN order_header oh ON od.ORDER_NUMBER = oh.ORDER_NUMBER
"""
fact_returns = pd.read_sql_query(query_returns, conn_sales)
fact_returns['RETURN_DATE'] = pd.to_datetime(fact_returns['RETURN_DATE'], errors='coerce')
fact_returns['DATE_ID'] = fact_returns['RETURN_DATE'].dt.strftime('%Y%m%d').fillna(0).astype(int)

fact_targets = df_targets.copy()
fact_targets['DATE_ID'] = (fact_targets['SALES_YEAR'].astype(str) +
                           fact_targets['SALES_PERIOD'].astype(str).str.zfill(2) + '01').astype(int)
fact_targets = fact_targets[['DATE_ID', 'SALES_STAFF_CODE', 'PRODUCT_NUMBER', 'RETAILER_CODE', 'SALES_TARGET']]

/tmp/ipykernel_10003/3763140214.py:18: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  fact_sales['ORDER_DATE'] = pd.to_datetime(fact_sales['ORDER_DATE'])


# 5. Data laden in het nieuwe Data Warehouse (GreatOutdoors_DWH.sqlite)

In [7]:
dwh_conn = sqlite3.connect('GreatOutdoors_DWH.sqlite')

dim_product.to_sql('Dim_Product', dwh_conn, if_exists='replace', index=False)
dim_customer.to_sql('Dim_Customer', dwh_conn, if_exists='replace', index=False)
dim_staff.to_sql('Dim_Staff', dwh_conn, if_exists='replace', index=False)
dim_return_reason.to_sql('Dim_Return_Reason', dwh_conn, if_exists='replace', index=False)
dim_date.to_sql('Dim_Date', dwh_conn, if_exists='replace', index=False)

fact_sales.to_sql('Fact_Sales', dwh_conn, if_exists='replace', index=False)
fact_returns.to_sql('Fact_Returns', dwh_conn, if_exists='replace', index=False)
fact_targets.to_sql('Fact_Sales_Targets', dwh_conn, if_exists='replace', index=False)

conn_crm.close()
conn_sales.close()
conn_staff.close()
dwh_conn.close()